# 01 — ReefUNet: Arquitectura e Treino

**Definição canónica** do modelo U-Net para segmentação de recifes em MDT-50cm.

Este notebook define a arquitectura, faz um sanity check e mostra como lançar o treino.
O treino real corre em `scripts/dgt_train_unet.py` (mais eficiente para runs longas).

| Aspecto | Valor |
|---|---|
| Input | (B, 1, 256, 256) — MDT normalizado [0, 1] |
| Output | (B, 1, 256, 256) — probabilidade sigmoid [0, 1] |
| Features | [64, 128, 256, 512] + bottleneck 1024 |
| Parâmetros | ~31 M |
| Treino | CPU-only (AMD EPYC-Milan 96 cores) |
| Loss | BCE ponderado (pos_weight=10) + Soft Dice |
| Optimizador | Adam lr=5e-4, weight_decay=1e-4 |
| Scheduler | CosineAnnealingLR T_max=100 |

In [ ]:
import sys, os
from pathlib import Path

WORK = Path('/home/jovyan/reef-imagery-pipeline')
if WORK.exists():
    sys.path.insert(0, str(WORK))
    os.chdir(str(WORK))

import torch
import torch.nn as nn

torch.set_num_threads(min(os.cpu_count() or 1, 16))
torch.manual_seed(42)

print(f'PyTorch: {torch.__version__}')
print(f'Threads: {torch.get_num_threads()}  |  Cores: {os.cpu_count()}')
print(f'Device: CPU-only (NUNCA .cuda() neste servidor)')

In [ ]:
# ── Arquitectura canónica ReefUNet ────────────────────────────────────────────
from src.ml_unet_model import ReefUNet, DoubleConv, get_loss_function, calculate_iou

model = ReefUNet()  # features=[64, 128, 256, 512] por defeito
print(f'Parâmetros: {model.num_parameters:,}  (~{model.num_parameters*4/1e6:.0f} MB)')
print()

# Estrutura do modelo
print('Encoder (downs):')
for i, d in enumerate(model.downs):
    c_in  = list(d.net.children())[0].in_channels
    c_out = list(d.net.children())[0].out_channels
    print(f'  down[{i}]: {c_in} → {c_out}')
print(f'Bottleneck: {list(model.bottleneck.net.children())[0].in_channels} → '
      f'{list(model.bottleneck.net.children())[0].out_channels}')
print('Decoder (ups): ConvTranspose2d + DoubleConv × 4')
print(f'Final:  1×1 Conv → sigmoid')

In [ ]:
# ── Sanity check: forward pass + loss + IoU ───────────────────────────────────
import numpy as np

model.eval()
criterion = get_loss_function(pos_weight=10.0)

# Simular batch de 2 tiles 256×256
x = torch.randn(2, 1, 256, 256)
y = (torch.rand(2, 1, 256, 256) > 0.94).float()  # ~6% de pixels reef (idem ao dataset)

with torch.no_grad():
    probs = model(x)

loss = criterion(probs, y)
iou  = calculate_iou(probs, y)

assert probs.shape == x.shape,         f'Shape errada: {probs.shape}'
assert 0.0 <= probs.min().item() <= 1.0
assert 0.0 <= probs.max().item() <= 1.0

print(f'Input:  {tuple(x.shape)}')
print(f'Output: {tuple(probs.shape)}  range [{probs.min():.3f}, {probs.max():.3f}]')
print(f'Loss (BCE+Dice, pos_weight=10): {loss.item():.4f}')
print(f'IoU (untrained):                {iou:.4f}')
print()
print('✓ Arquitectura OK — sigmoid aplicado internamente em forward()')
print('✓ Loss e IoU funcionam com probabilidades [0,1]')

In [ ]:
# ── Desequilíbrio de classes — porque o WeightedRandomSampler é crítico ───────
import matplotlib.pyplot as plt

# Dataset: 341 patches, 19 com recife (5.6%), 322 vazios (94.4%)
n_reef   = 19
n_empty  = 322
n_total  = n_reef + n_empty

# Sem WeightedRandomSampler: batch típico de 8 tem ~0.5 patches com recife
# Com WeightedRandomSampler (reef×18): batch tem ~4 patches com recife
weight_reef  = 18.0
weight_empty = 1.0
eff_reef_pct = (n_reef * weight_reef) / (n_reef * weight_reef + n_empty * weight_empty) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Antes: distribuição original
axes[0].bar(['Vazio (322)', 'Recife (19)'], [n_empty, n_reef],
            color=['#4a90d9', '#e74c3c'], edgecolor='k')
axes[0].set_title('Dataset Original\n(sem oversample)')
axes[0].set_ylabel('Nº de patches')
axes[0].text(1, n_reef + 5, f'{n_reef/n_total*100:.1f}%', ha='center', fontsize=12, color='red')

# Depois: distribuição efectiva com WeightedRandomSampler
eff_reef  = n_reef  * weight_reef
eff_empty = n_empty * weight_empty
axes[1].bar(['Vazio', 'Recife (×18)'], [eff_empty, eff_reef],
            color=['#4a90d9', '#27ae60'], edgecolor='k')
axes[1].set_title(f'Com WeightedRandomSampler (reef×{weight_reef:.0f})\n≈{eff_reef_pct:.0f}% patches com recife por batch')
axes[1].set_ylabel('Peso efectivo de amostragem')
axes[1].text(1, eff_reef + 5, f'{eff_reef_pct:.0f}%', ha='center', fontsize=12, color='green')

plt.suptitle('Desequilíbrio de Classes — Antes e Depois do Fix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Sem sampler: batch de 8 tem ~{8*n_reef/n_total:.1f} patches com recife')
print(f'Com sampler: batch de 8 tem ~{8*eff_reef_pct/100:.1f} patches com recife')

In [ ]:
# ── Lançar treino no HPC ──────────────────────────────────────────────────────
# O treino real corre em scripts/dgt_train_unet.py (mais eficiente).
# Este bloco é apenas para verificar que o ambiente está pronto.

from pathlib import Path
import json

MODELS_DIR   = WORK / 'models'
DATASET_DIR  = WORK / 'data' / 'master_ml_dataset'
TRAIN_SCRIPT = WORK / 'scripts' / 'dgt_train_unet.py'
LOG_FILE     = WORK / 'training_log.json'
BEST_WEIGHTS = MODELS_DIR / 'unet_reef_best.pth'

print('=== Verificação de ambiente ===')
print(f'Dataset dir:   {DATASET_DIR.exists()} — {DATASET_DIR}')
print(f'Train script:  {TRAIN_SCRIPT.exists()} — {TRAIN_SCRIPT}')
print(f'Best weights:  {BEST_WEIGHTS.exists()} — {BEST_WEIGHTS}')

if DATASET_DIR.exists():
    imgs  = list((DATASET_DIR / 'images').glob('*.tif'))
    masks = list((DATASET_DIR / 'masks').glob('*.tif'))
    print(f'Patches:       {len(imgs)} imagens / {len(masks)} máscaras')

if LOG_FILE.exists():
    with open(str(LOG_FILE)) as f:
        history = json.load(f)
    last = history[-1]
    print(f'Treino:        {len(history)} epochs  best_iou={last["best_iou"]:.4f}  '
          f'(última epoch: vl_loss={last["val_loss"]:.4f}  iou={last["val_iou"]:.4f})')
else:
    print('Treino:        ainda não iniciado')

print()
print('Para iniciar / continuar treino:')
print(f'  !python {TRAIN_SCRIPT}')
print()
print('Nota: ~80 s/epoch com ReefUNet [64,128,256,512] em 96 cores CPU.')
print('100 epochs ≈ 133 min. Resumível (checkpoint_latest.pth guardado por epoch).')

In [ ]:
# ── Visualizar curvas de treino (se treino já correu) ─────────────────────────
import matplotlib.pyplot as plt
import json
from pathlib import Path

log_path = WORK / 'training_log.json'

if not log_path.exists():
    print('training_log.json não encontrado — corra o treino primeiro.')
else:
    with open(str(log_path)) as f:
        history = json.load(f)

    epochs     = [r['epoch']      for r in history]
    tr_loss    = [r['train_loss'] for r in history]
    vl_loss    = [r['val_loss']   for r in history]
    vl_iou     = [r['val_iou']    for r in history]
    best_iou   = [r['best_iou']   for r in history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, tr_loss, label='Train loss', color='#3498db')
    axes[0].plot(epochs, vl_loss, label='Val loss',   color='#e74c3c')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (BCE + Dice)')
    axes[0].set_title('Curva de Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, vl_iou,   label='Val IoU',   color='#27ae60')
    axes[1].plot(epochs, best_iou, label='Best IoU',  color='#f39c12', linestyle='--')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('IoU')
    axes[1].set_title('Curva de IoU (Jaccard)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(f'ReefUNet v3 — {len(history)} epochs  best_iou={max(best_iou):.4f}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'Epochs concluídas: {len(history)}')
    print(f'Melhor IoU: {max(best_iou):.4f}  na epoch {best_iou.index(max(best_iou))+1}')
    print(f'Val loss final: {vl_loss[-1]:.4f}')